In [1]:
import duckdb

db = duckdb.connect('datahmw/stations.db')
db.sql("""
    CREATE TABLE IF NOT EXISTS stations AS 
    SELECT * FROM read_csv_auto('datahmw/stations-2023-09.csv')
""")



In [2]:
db.sql("""
    CREATE TABLE IF NOT EXISTS distances AS 
    SELECT * FROM read_csv_auto('datahmw/tariff-distances-2022-01.csv',
       nullstr = 'XXX')
""")

db.sql("""CREATE TABLE IF NOT EXISTS distances_long AS
    UNPIVOT distances
    ON COLUMNS (* EXCLUDE station)
    INTO NAME other_station VALUE distance;""")

db.sql("""SELECT station, other_station, distance
FROM distances_long
LIMIT 3;""")

db.sql("""SELECT
    s1.name_long AS station1,
    s2.name_long AS station2,
    distances_long.distance
FROM distances_long
JOIN stations s1 ON distances_long.station = s1.code
JOIN stations s2 ON distances_long.other_station = s2.code
WHERE s1.country = 'NL'
  AND s2.country = 'NL'
  AND station < other_station
ORDER BY distance DESC
LIMIT 3;""")


┌──────────────────┬────────────────────┬──────────┐
│     station1     │      station2      │ distance │
│     varchar      │      varchar       │  int64   │
├──────────────────┼────────────────────┼──────────┤
│ Eemshaven        │ Vlissingen         │      426 │
│ Bad Nieuweschans │ Vlissingen         │      425 │
│ Eemshaven        │ Vlissingen Souburg │      425 │
└──────────────────┴────────────────────┴──────────┘

In [ ]:
import duckdb

db.sql("""INSTALL postgres; LOAD postgres;""")
conn_string = "host=localhost user=postgres password=postgres dbname=postgres"
db.sql(f"""
ATTACH IF NOT EXISTS '{conn_string}' AS postgres_db (TYPE postgres)
""")
db.sql("""
    CREATE OR REPLACE TABLE disruptions AS
    SELECT * FROM read_csv('datahmw/disruptions-202*.csv')
""")


In [ ]:
# zmniejszylem dataset do lat 2019, 2023, 2024, 2025

db.sql("""
COPY (SELECT * FROM "datahmw/services-20*.csv") TO "datahmw/services.parquet" (FORMAT parquet, COMPRESSION 'ZSTD');
""")

db.sql("""
    CREATE OR REPLACE TABLE services AS 
    SELECT * FROM 'datahmw/services.parquet'
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
#1 How many trains departed from Amsterdam Central station overall?

db.sql("""SELECT count(*) FROM services 
              WHERE "Stop:Arrival time" IS NULL
        AND "Stop:Station name" = 'Amsterdam Centraal'
       """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       607405 │
└──────────────┘

In [6]:
#2 Calculate the average arrival delay of different service types (Service:Type). Order results descending by average delay.


db.sql("""
SELECT "Service:Type",
       AVG("Stop:Arrival delay") AS avg_arrival_delay
FROM services
WHERE "Stop:Arrival delay" IS NOT NULL
GROUP BY "Service:Type"
ORDER BY avg_arrival_delay DESC
""")

┌──────────────────────┬────────────────────┐
│     Service:Type     │ avg_arrival_delay  │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ Alpen Express        │  47.68627450980392 │
│ Krokus Express       │ 15.916666666666666 │
│ European Sleeper     │ 14.262589044460821 │
│ Nightjet             │   9.37664644714038 │
│ Stoomtrein           │  7.785714285714286 │
│ ICE                  │   7.76692176533566 │
│ Eurostar             │   7.58370976627056 │
│ ICE International    │  6.484149152347649 │
│ Thalys               │ 6.4302223522527795 │
│ Nachttrein           │  3.675754189944134 │
│     ·                │                 ·  │
│     ·                │                 ·  │
│     ·                │                 ·  │
│ Taxibus ipv trein    │                0.0 │
│ Metro ipv trein      │                0.0 │
│ Niet instappen       │                0.0 │
│ Metro                │                0.0 │
│ stoptrein            │          

In [7]:
#3 What was the most common disruption cause in different years? MODE function may be useful.
db.sql("""
SELECT mode(cause_en) FROM disruptions
""")


┌───────────────────┐
│ "mode"(cause_en)  │
│      varchar      │
├───────────────────┤
│ broken down train │
└───────────────────┘

In [8]:
#4 How many trains started their overall service in any Amsterdam station?

db.sql("""
SELECT COUNT(*) "Stop:RDT-ID"
FROM services WHERE "Stop:Station name" ILIKE 'amsterdam%' AND "Stop:Arrival time" IS NULL
""")


┌─────────────┐
│ Stop:RDT-ID │
│    int64    │
├─────────────┤
│      708066 │
└─────────────┘

In [ ]:
#5 What fraction of services was run to final destinations outside the Netherlands?

db.sql("""
SELECT 
    COUNT(*) FILTER (WHERE st.country != 'NL') * 1.0 / COUNT(*) AS fraction_outside_nl
FROM services s
JOIN stations st ON s."Stop:Station code" = st.code
WHERE s."Stop:Departure time" IS NULL
""")


┌─────────────────────┐
│ fraction_outside_nl │
│       double        │
├─────────────────────┤
│ 0.03646645886669597 │
└─────────────────────┘

In [ ]:
#6 What is the largest distance between stations in the Netherlands (code NL)?

db.sql("""SELECT 
    s1.name_short AS stacja_poczatkowa, 
    s2.name_short AS stacja_docelowa, 
    d.distance AS dystans_km
FROM distances_long d
JOIN stations s1 ON d.Station = s1.code
JOIN stations s2 ON d.other_station = s2.code
WHERE s1.country = 'NL' AND s2.country = 'NL'
ORDER BY d.distance DESC
LIMIT 1
       """)


┌───────────────────┬─────────────────┬────────────┐
│ stacja_poczatkowa │ stacja_docelowa │ dystans_km │
│      varchar      │     varchar     │   int64    │
├───────────────────┼─────────────────┼────────────┤
│ Eemshaven         │ Vlissingen      │        426 │
└───────────────────┴─────────────────┴────────────┘